# Ajustes tardios da Orbe

O arquivo adicional precisa entrar no fechamento sem duplicar movimentos nem tratar a taxa cobrada pela adquirente como comissão da Tarvo. Este notebook compara o lote com o relatório original, valida referências e mede o impacto incremental.

A referência de negócio é [docs/documentação-geral.md](../docs/documentação-geral.md), versão das 14h disponível após a integração da develop. O hash desse documento é registrado junto aos hashes dos dados. A RN-65 confirma a inclusão dos ajustes na competência da data_movimento; o RF-22 exige reconhecer todos os relatórios Orbe pelo cabeçalho.

Fontes locais: relatório original, adicional e planilha de contratos. Downloads e arquivos de entrada são preservados. Esta análise substitui a conclusão anterior de que faltava a numeração das regras; não altera o motor nem publica CSVs finais.

In [1]:
from pathlib import Path
from decimal import Decimal, ROUND_HALF_UP
from datetime import datetime, date
from collections import Counter
import csv, io, re, hashlib, unicodedata
import pandas as pd
from IPython.display import display

RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'docs/FORMATO_SAIDA.md').exists())
PASTA = RAIZ/'Tarvo_Data_Pack_set2026'  # altere para o pacote desejado

def unico(padrao):
    encontrados = sorted(PASTA.glob(padrao))
    assert len(encontrados)==1, (padrao,encontrados)
    return encontrados[0]
ORIGINAL, AJUSTES, TAXAS = unico('01_orbe_*.csv'), unico('06_orbe_*.csv'), unico('02_*.xlsx')
CONTEXTO = RAIZ/'docs/documentação-geral.md'
assert CONTEXTO.exists(), 'Documento de regras do repositório ausente.'
FONTES=[ORIGINAL,AJUSTES,TAXAS,CONTEXTO]
hashes={p:hashlib.sha256(p.read_bytes()).hexdigest() for p in FONTES}
texto_regras=CONTEXTO.read_text(encoding='utf-8')

def ler_csv(p):
    raw=p.read_bytes()
    try: texto=raw.decode('utf-8-sig'); encoding='utf-8-sig'
    except UnicodeDecodeError: texto=raw.decode('cp1252'); encoding='cp1252'
    reader=csv.DictReader(io.StringIO(texto),delimiter=';')
    rows=[]
    for r in reader:
        assert None not in r and all(v is not None for v in r.values()), (p,r)
        assert r['id_transacao'].strip(), (p,r)
        rows.append(dict(r,arquivo_origem=p.name,ref_origem=r['id_transacao']))
    return rows,reader.fieldnames,encoding
base,cols_base,enc_base=ler_csv(ORIGINAL)
lote,cols_lote,enc_lote=ler_csv(AJUSTES)
assert cols_base==cols_lote
campos_dinheiro=['valor_bruto','valor_taxa','valor_liquido']
def decimal_br(s):
    assert re.fullmatch(r'-?(?:\d{1,3}(?:\.\d{3})+|\d+),\d{2}',s), s
    return Decimal(s.replace('.','').replace(',','.'))
def data_br(s): return datetime.strptime(s,'%d/%m/%Y').date()
def total(v): return sum(v,Decimal('0.00'))
def centavos(v): return v.quantize(Decimal('0.01'),rounding=ROUND_HALF_UP)
for r in base+lote:
    for k in campos_dinheiro: r[k]=decimal_br(r[k])
    r['taxa_aplicada_pct']=Decimal(r['taxa_aplicada_pct'].replace(',','.'))
    r['data_movimento']=data_br(r['data_movimento'])
    r['data_venda_original']=data_br(r['data_venda_original']) if r['data_venda_original'] else None
print('Base:',len(base),'| adicional:',len(lote))
display(pd.DataFrame([{'arquivo':p.name,'sha256':hashes[p]} for p in FONTES]))
print('Encodings:',enc_base,enc_lote,'| delimitador: ;')
display(pd.DataFrame(lote))
display(pd.DataFrame(lote).groupby('tipo_lancamento').agg(quantidade=('id_transacao','size'),bruto=('valor_bruto',total),taxa_observada=('valor_taxa',total),liquido=('valor_liquido',total)))


Base: 1943 | adicional: 14


,arquivo,sha256
0,01_orbe_relatorio_comissoes_set2026.csv,1d9891971da84f11fb1f25e0d30b0b8633865653bb9354...
1,06_orbe_ajustes_tardios_set2026.csv,b1cfd183945fb194e05cfbe9455ab364c31b676427e665...
2,02_tabela_taxas_contratadas.xlsx,d0c7d2b13ac41c2a036a40362eb26c341666feef74d719...
3,documentação-geral.md,7a3a2acf196364613e460d4a1035bdff989c2866a861a8...


Encodings: utf-8-sig utf-8-sig | delimitador: ;


,id_transacao,data_movimento,cod_estabelecimento,nome_estabelecimento,tipo_lancamento,modalidade,qtd_parcelas,bandeira,valor_bruto,taxa_aplicada_pct,valor_taxa,valor_liquido,id_referencia,data_venda_original,arquivo_origem,ref_origem
0,TX88446992,2026-09-30,1014537966,DUBLIN HUB INTERCAMBIO,ESTORNO,CREDITO A VISTA,1,VISA,-1147.05,2.59,-29.70,-1117.35,TX88431689,2026-09-18,06_orbe_ajustes_tardios_set2026.csv,TX88446992
1,TX88447011,2026-09-30,1041180753,MAPLE STUDY ASSESSORIA,ESTORNO,DEBITO,1,VISA,-420.77,1.49,-6.26,-414.51,TX88413573,2026-09-03,06_orbe_ajustes_tardios_set2026.csv,TX88447011
2,TX88447030,2026-09-30,1037647039,Mundo Aberto,ESTORNO,CREDITO A VISTA,1,VISA,-562.44,2.49,-14.00,-548.44,TX88438558,2026-09-25,06_orbe_ajustes_tardios_set2026.csv,TX88447030
3,TX88447044,2026-09-30,1075624862,BRIDGE IDIOMAS,ESTORNO,CREDITO PARCELADO,2,ELO,-1982.10,3.19,-63.22,-1918.88,TX88425142,2026-09-13,06_orbe_ajustes_tardios_set2026.csv,TX88447044
4,TX88447057,2026-09-30,1020660828,EDU ABROAD CONSULTORIA,ESTORNO,CREDITO PARCELADO,2,ELO,-3898.29,3.19,-124.35,-3773.94,TX88430977,2026-09-18,06_orbe_ajustes_tardios_set2026.csv,TX88447057
5,TX88447070,2026-09-30,1075624862,BRIDGE IDIOMAS,ESTORNO,CREDITO PARCELADO,6,VISA,-2603.31,3.19,-83.04,-2520.27,TX88424537,2026-09-12,06_orbe_ajustes_tardios_set2026.csv,TX88447070
6,TX88447087,2026-09-30,1000960410,OXFORD PREP CURSOS,ESTORNO,CREDITO PARCELADO,9,AMEX,-8647.36,3.79,-327.73,-8319.63,TX88422829,2026-09-11,06_orbe_ajustes_tardios_set2026.csv,TX88447087
7,TX88447093,2026-09-30,1041180753,MAPLE STUDY ASSESSORIA,ESTORNO,CREDITO A VISTA,1,AMEX,-2620.34,2.69,-70.48,-2549.86,TX88416714,2026-09-06,06_orbe_ajustes_tardios_set2026.csv,TX88447093
8,TX88447109,2026-09-30,1020660828,EDU ABROAD CONSULTORIA,ESTORNO,CREDITO PARCELADO,3,ELO,-2747.67,3.19,-87.65,-2660.02,TX88428997,2026-09-16,06_orbe_ajustes_tardios_set2026.csv,TX88447109
9,TX88447140,2026-09-30,1000960410,OXFORD PREP CURSOS,ESTORNO,CREDITO A VISTA,1,VISA,-2366.18,2.59,-61.28,-2304.90,TX88432148,2026-09-19,06_orbe_ajustes_tardios_set2026.csv,TX88447140


,quantidade,bruto,taxa_observada,liquido
tipo_lancamento,,,,
ESTORNO,10,-26995.51,-867.71,-26127.80
REVERSAO ANTECIPACAO,3,-124353.87,-2611.42,-121742.45
VENDA,1,5480.00,174.81,5305.19


## Identidade, referências e conservação

O mesmo ID em dois arquivos pode ser reenvio ou conflito. Nenhum caso é removido silenciosamente. A verificação distingue duplicata interna, repetição entre lotes e mesmo ID com conteúdo diferente. Referências são verificadas também contra o lote adicional para suportar dependências dentro da mesma entrega.

In [2]:
indice={}
for r in base+lote: indice.setdefault(r['id_transacao'],[]).append(r)
def assinatura(r): return tuple(r[k] for k in cols_base)
repetidos=[{'id_transacao':k,'ocorrencias':len(v),'conteudos':len({assinatura(r) for r in v}),
             'status':'REENVIO_IDENTICO' if len({assinatura(r) for r in v})==1 else 'CONFLITO'}
            for k,v in indice.items() if len(v)>1]
print('Duplicatas internas base:',len(base)-len({r['id_transacao'] for r in base}))
print('Duplicatas internas adicional:',len(lote)-len({r['id_transacao'] for r in lote}))
print('IDs do adicional presentes na base:',len({r['id_transacao'] for r in lote}&{r['id_transacao'] for r in base}))
display(pd.DataFrame(repetidos,columns=['id_transacao','ocorrencias','conteudos','status']))
referencias=[]
for r in lote:
    ref=r['id_referencia']; candidatos=indice.get(ref,[]) if ref else []
    orig=candidatos[0] if len(candidatos)==1 else None
    reversao=r['tipo_lancamento']=='REVERSAO ANTECIPACAO'
    esperado='ANTECIPACAO' if reversao else 'VENDA'
    referencias.append(dict(id_transacao=r['id_transacao'],id_referencia=ref,
        qtd_origens=len(candidatos),arquivo_referenciado=orig['arquivo_origem'] if orig else '',
        mesmo_ec=orig['cod_estabelecimento']==r['cod_estabelecimento'] if orig else None,
        tipo_coerente=orig['tipo_lancamento']==esperado if orig else None,
        data_coerente=orig['data_movimento']==r['data_venda_original'] if orig else None,
        data_nao_futura=orig['data_movimento']<=r['data_movimento'] if orig else None,
        modalidade_coerente=orig['modalidade']==r['modalidade'] if orig else None,
        parcelas_coerentes=orig['qtd_parcelas']==r['qtd_parcelas'] if orig else None,
        bruto_oposto=orig['valor_bruto']==-r['valor_bruto'] if orig else None,
        taxa_oposta=orig['valor_taxa']==-r['valor_taxa'] if orig else None,
        liquido_oposto=orig['valor_liquido']==-r['valor_liquido'] if orig else None))
refs=pd.DataFrame(referencias)
display(refs)
for r in lote:
    assert r['valor_bruto']-r['valor_taxa']==r['valor_liquido'],r['id_transacao']
    if r['tipo_lancamento'] in ['ESTORNO','REVERSAO ANTECIPACAO']:
        assert r['valor_bruto']<0 and r['id_referencia'],r['id_transacao']
    elif r['tipo_lancamento']=='VENDA': assert r['valor_bruto']>0,r['id_transacao']
print('Campos vazios do lote (vazios de modalidade são esperados nas reversões):')
display(pd.DataFrame(lote).apply(lambda s:s.map(lambda v:v is None or v=='').sum()).to_frame('vazios'))
# Limite acumulado: estornos anteriores e novos podem consumir o mesmo original.
consumos=[]
for ref in sorted({r['id_referencia'] for r in lote if r['id_referencia']}):
    origens=indice.get(ref,[])
    ajustes=[r for r in base+lote if r['id_referencia']==ref]
    if len(origens)==1:
        origem=origens[0]
        consumos.append(dict(ref_origem=ref,ajustes=len(ajustes),valor_original=origem['valor_bruto'],
            valor_revertido=-total(r['valor_bruto'] for r in ajustes),
            excede_original=-total(r['valor_bruto'] for r in ajustes)>abs(origem['valor_bruto'])))
display(pd.DataFrame(consumos))


Duplicatas internas base: 0
Duplicatas internas adicional: 0
IDs do adicional presentes na base: 0


,id_transacao,ocorrencias,conteudos,status


,id_transacao,id_referencia,qtd_origens,arquivo_referenciado,mesmo_ec,tipo_coerente,data_coerente,data_nao_futura,modalidade_coerente,parcelas_coerentes,bruto_oposto,taxa_oposta,liquido_oposto
0,TX88446992,TX88431689,1,01_orbe_relatorio_comissoes_set2026.csv,True,True,True,True,True,True,True,True,True
1,TX88447011,TX88413573,1,01_orbe_relatorio_comissoes_set2026.csv,True,True,True,True,True,True,True,True,True
2,TX88447030,TX88438558,1,01_orbe_relatorio_comissoes_set2026.csv,True,True,True,True,True,True,True,True,True
3,TX88447044,TX88425142,1,01_orbe_relatorio_comissoes_set2026.csv,True,True,True,True,True,True,True,True,True
4,TX88447057,TX88430977,1,01_orbe_relatorio_comissoes_set2026.csv,True,True,True,True,True,True,True,True,True
5,TX88447070,TX88424537,1,01_orbe_relatorio_comissoes_set2026.csv,True,True,True,True,True,True,True,True,True
6,TX88447087,TX88422829,1,01_orbe_relatorio_comissoes_set2026.csv,True,True,True,True,True,True,True,True,True
7,TX88447093,TX88416714,1,01_orbe_relatorio_comissoes_set2026.csv,True,True,True,True,True,True,True,True,True
8,TX88447109,TX88428997,1,01_orbe_relatorio_comissoes_set2026.csv,True,True,True,True,True,True,True,True,True
9,TX88447140,TX88432148,1,01_orbe_relatorio_comissoes_set2026.csv,True,True,True,True,True,True,True,True,True


Campos vazios do lote (vazios de modalidade são esperados nas reversões):


,vazios
id_transacao,0
data_movimento,0
cod_estabelecimento,0
nome_estabelecimento,0
tipo_lancamento,0
modalidade,3
qtd_parcelas,3
bandeira,3
valor_bruto,0
taxa_aplicada_pct,0


,ref_origem,ajustes,valor_original,valor_revertido,excede_original
0,AN88446397,1,46080.41,46080.41,False
1,AN88446551,1,36583.37,36583.37,False
2,AN88446748,1,41690.09,41690.09,False
3,TX88413573,1,420.77,420.77,False
4,TX88416714,1,2620.34,2620.34,False
5,TX88422829,1,8647.36,8647.36,False
6,TX88424537,1,2603.31,2603.31,False
7,TX88425142,1,1982.10,1982.10,False
8,TX88428997,1,2747.67,2747.67,False
9,TX88430977,1,3898.29,3898.29,False


## Regras e impacto potencial

RN-08 e RN-09: comissão pela taxa contratada, Decimal e arredondamento por linha. RN-10 a RN-13: a faixa vem da própria linha, a vigência vem da venda original e o estorno abate no mês do estorno. Modalidade ou parcelas diferentes da venda são evidências para inspeção, sem contrariar a faixa do próprio estorno. RN-14: a reversão anula somente a antecipação referenciada.

A leitura resolve contratos por EC e vigência explícita. Um EC ausente na planilha só é ligado por declaração da RN-05, usando nome exato normalizado; não há similaridade de nomes. Vínculos vêm do documento, sem IDs do mês fixados na lógica. Contrato ausente ou ambíguo gera pendência ao comercial.

Valores observados da Orbe, cálculos contratuais verificados e aprovação do fechamento permanecem separados. A RN-65 confirma que o adicional pertence ao fechamento; a chegada tardia, isoladamente, não constitui pendência ou motivo de bloqueio.

In [3]:
def norm(s):
    return re.sub(r'[^A-Z0-9 ]','', ''.join(c for c in unicodedata.normalize('NFKD',str(s).upper()) if not unicodedata.combining(c))).strip()
contratos=pd.read_excel(TAXAS,sheet_name='Taxas vigentes',header=4,dtype=str).fillna('')
contratos=contratos[contratos.CNPJ.ne('')].copy()
# Declarações EC = nome no contexto (seção de confirmação do financeiro).
secao=next(linha for linha in texto_regras.splitlines() if '| RN-05 |' in linha)
declaracoes=re.findall(r'(\d{10}) = ([^;.\n]+)',secao)
mapa_contexto={norm(re.sub(r'\([^)]*\)','',nome)):ec for ec,nome in declaracoes}
contratos['ec']=contratos['EC Orbe'].map(lambda s:str(int(Decimal(s))) if s else '')
for i,r in contratos.iterrows():
    if not r['ec'] and norm(r['Lojista']) in mapa_contexto:
        contratos.at[i,'ec']=mapa_contexto[norm(r['Lojista'])]

def vigencia(s):
    datas=re.findall(r'\d{2}/\d{2}/\d{4}',s)
    if len(datas)!=1: return None
    d=data_br(datas[0]); original=s.strip(); s=norm(s)
    if s.startswith('ATE '): return date.min,d
    if s.startswith('A PARTIR DE '): return d,date.max
    if re.fullmatch(r'\d{2}/\d{2}/\d{4}',original): return d,date.max
    return None

def percentual(s):
    s=str(s).strip()
    return Decimal(s[:-1].replace(',','.'))/100 if s.endswith('%') else Decimal(s.replace(',','.'))

def faixa(r):
    if r['tipo_lancamento'] in ['ANTECIPACAO','REVERSAO ANTECIPACAO']: return 'Antecipação'
    if r['modalidade'] in ['DEBITO','PIX','CREDITO A VISTA']:
        return {'DEBITO':'Débito','PIX':'Pix','CREDITO A VISTA':'Crédito à vista'}[r['modalidade']]
    if r['modalidade']=='CREDITO PARCELADO' and r['qtd_parcelas'].isdigit():
        q=int(r['qtd_parcelas'])
        if 2<=q<=6: return 'Créd. 2-6x'
        if 7<=q<=12: return 'Créd. 7-12x'
    return None

def contrato_para(r):
    d=r['data_venda_original'] if r['tipo_lancamento'] in ['ESTORNO','REVERSAO ANTECIPACAO'] else r['data_movimento']
    f=faixa(r)
    candidatos=[]
    for _,t in contratos.iterrows():
        v=vigencia(t['Vigência'])
        if t['ec']==r['cod_estabelecimento'] and v and d and v[0]<=d<=v[1] and 'INATIVO' not in norm(t['Obs']):
            candidatos.append(t)
    if len(candidatos)!=1 or not f or not candidatos[0][f]: return None,'Contrato/faixa/vigência ausente ou ambígua'
    return percentual(candidatos[0][f]),'Contrato vigente; EC explícito na planilha ou contexto'

resultados=[]; pendencias=[]
for r in lote:
    taxa,fonte=contrato_para(r)
    ref=next(x for x in referencias if x['id_transacao']==r['id_transacao'])
    problemas=[]
    if len(indice[r['id_transacao']])!=1: problemas.append('ID repetido/conflitante')
    if r['id_referencia']:
        # RN-10: modalidade/parcelas vêm do próprio estorno; diferenças são evidência, não bloqueio automático.
        campos=['mesmo_ec','tipo_coerente','data_coerente','data_nao_futura']
        if ref['qtd_origens']!=1 or not all(ref[k] for k in campos): problemas.append('Referência ausente, ambígua ou incompatível')
        if any(x['excede_original'] for x in consumos if x['ref_origem']==r['id_referencia']): problemas.append('Reversões acumuladas excedem original')
    if taxa is None: problemas.append(fonte)
    comissao=centavos(r['valor_bruto']*taxa) if taxa is not None and not problemas else None
    # Reversão deve anular a comissão da antecipação original, inclusive arredondamento.
    if comissao is not None and r['tipo_lancamento']=='REVERSAO ANTECIPACAO':
        orig=indice[r['id_referencia']][0]; taxa_orig,_=contrato_para(orig)
        if taxa_orig is None or comissao != -centavos(orig['valor_bruto']*taxa_orig):
            problemas.append('Reversão não anula comissão da antecipação original'); comissao=None
    dif=centavos(r['valor_bruto']*(r['taxa_aplicada_pct']/100-taxa)) if comissao is not None else None
    resultados.append(dict(id_transacao=r['id_transacao'],tipo=r['tipo_lancamento'],ec=r['cod_estabelecimento'],
        data_ref_contrato=r['data_venda_original'] or r['data_movimento'],
        valor_taxa_observado=r['valor_taxa'],taxa_contratada=taxa,comissao_incremental=comissao,
        divergencia_taxa_incremental=dif,base_confirmacao='RN-08 a RN-14 / RN-65 + contrato' if comissao is not None else 'PENDENTE',
        arquivo_origem=r['arquivo_origem'],ref_origem=r['ref_origem']))
    for motivo in problemas:
        pendencias.append(dict(tipo='AJUSTE_TARDIO',ref_origem=r['ref_origem'],valor_brl=r['valor_bruto'],motivo=motivo,
            quem_decide='COMERCIAL' if 'Contrato' in motivo else 'FINANCEIRO',proximo_passo='Validar fonte/contrato antes da liberação'))
res=pd.DataFrame(resultados)
display(res)
display(res.groupby('tipo').agg(linhas=('id_transacao','size'),taxa_observada=('valor_taxa_observado',total),
    comissao_calculada=('comissao_incremental',lambda s:total(v for v in s if v is not None)),
    divergencia_calculada=('divergencia_taxa_incremental',lambda s:total(v for v in s if v is not None))))
print('Comissão incremental calculável:',total(r['comissao_incremental'] for r in resultados if r['comissao_incremental'] is not None))
print('Valor de taxa observado (não é comissão):',total(r['valor_taxa'] for r in lote))
print('Comissão sem cálculo seguro:',sum(r['comissao_incremental'] is None for r in resultados),'linhas')
display(res.groupby('ec').agg(linhas=('id_transacao','size'),comissao_incremental=('comissao_incremental',lambda s:total(v for v in s if v is not None))))


,id_transacao,tipo,ec,data_ref_contrato,valor_taxa_observado,taxa_contratada,comissao_incremental,divergencia_taxa_incremental,base_confirmacao,arquivo_origem,ref_origem
0,TX88446992,ESTORNO,1014537966,2026-09-18,-29.70,0.0259,-29.71,-0.00,RN-08 a RN-14 / RN-65 + contrato,06_orbe_ajustes_tardios_set2026.csv,TX88446992
1,TX88447011,ESTORNO,1041180753,2026-09-03,-6.26,0.0149,-6.27,-0.00,RN-08 a RN-14 / RN-65 + contrato,06_orbe_ajustes_tardios_set2026.csv,TX88447011
2,TX88447030,ESTORNO,1037647039,2026-09-25,-14.00,0.0249,-14.00,-0.00,RN-08 a RN-14 / RN-65 + contrato,06_orbe_ajustes_tardios_set2026.csv,TX88447030
3,TX88447044,ESTORNO,1075624862,2026-09-13,-63.22,0.0319,-63.23,-0.00,RN-08 a RN-14 / RN-65 + contrato,06_orbe_ajustes_tardios_set2026.csv,TX88447044
4,TX88447057,ESTORNO,1020660828,2026-09-18,-124.35,0.0319,-124.36,-0.00,RN-08 a RN-14 / RN-65 + contrato,06_orbe_ajustes_tardios_set2026.csv,TX88447057
5,TX88447070,ESTORNO,1075624862,2026-09-12,-83.04,0.0319,-83.05,-0.00,RN-08 a RN-14 / RN-65 + contrato,06_orbe_ajustes_tardios_set2026.csv,TX88447070
6,TX88447087,ESTORNO,1000960410,2026-09-11,-327.73,0.0379,-327.73,-0.00,RN-08 a RN-14 / RN-65 + contrato,06_orbe_ajustes_tardios_set2026.csv,TX88447087
7,TX88447093,ESTORNO,1041180753,2026-09-06,-70.48,0.0269,-70.49,-0.00,RN-08 a RN-14 / RN-65 + contrato,06_orbe_ajustes_tardios_set2026.csv,TX88447093
8,TX88447109,ESTORNO,1020660828,2026-09-16,-87.65,0.0319,-87.65,-0.00,RN-08 a RN-14 / RN-65 + contrato,06_orbe_ajustes_tardios_set2026.csv,TX88447109
9,TX88447140,ESTORNO,1000960410,2026-09-19,-61.28,0.0259,-61.28,-0.00,RN-08 a RN-14 / RN-65 + contrato,06_orbe_ajustes_tardios_set2026.csv,TX88447140


,linhas,taxa_observada,comissao_calculada,divergencia_calculada
tipo,,,,
ESTORNO,10,-867.71,-867.77,0.00
REVERSAO ANTECIPACAO,3,-2611.42,-2611.43,0.00
VENDA,1,174.81,174.81,0.00


Comissão incremental calculável: -3304.39
Valor de taxa observado (não é comissão): -3304.32
Comissão sem cálculo seguro: 0 linhas


,linhas,comissao_incremental
ec,,
1000960410,4,-2232.19
1014537966,1,-29.71
1020660828,2,-212.01
1037647039,1,-14.00
1038867654,1,174.81
1041180753,3,-845.01
1075624862,2,-146.28


### Cobertura das regras solicitadas

| Regra | Evidência / teste | Conclusão |
|---|---|---|
| RN-13 | Estornos com referência, EC, data original e competência de data_movimento | Aplicável e verificada nas linhas do adicional |
| RN-14 | Reversões com origem do tipo antecipação, valores opostos e comissão anulada | Aplicável e verificada |
| RN-20 | Comparação das taxas aplicada e contratada; diferença monetária calculada por linha | Verificada; truncamento não gera divergência |
| RN-21 | Cada estorno mantém seu ID e contribuição assinada | Não há nova taxa errada neste lote; cenário sintético verifica estorno negativo |
| RN-22 | Soma das contribuições de venda e estorno | Pares reais e cenário de taxa errada verificados |
| RN-23 | Taxa aplicada menor que contratada é aceita como divergência negativa | Sem ocorrência real nova; comportamento testado com cenário sintético |
| RN-05 / RN-06 | Vínculo aprovado para EC ausente, sem similaridade de nomes | Conta contratual da venda nova resolvida |
| RN-08 a RN-12 | Contrato, arredondamento, faixa e vigência | Aplicadas ao lote; faixas fora de 12x não são calculadas |
| RN-65 / RF-22 | Inclusão dos ajustes, identidade, referências e detecção pelo cabeçalho | Verificadas na união e no ensaio de reenvio |
| RN-42 / RN-43 | Notas dependem do saldo mensal por lojista | Requisito documentado; emissão final fora deste notebook |
| RN-58 a RN-62 / D-13 / D-14 | Pendências, aprovação, situação e competência | Sem bloqueio artificial pela chegada do adicional; liberação com financeiro |

Verificação de um cenário sintético demonstra comportamento da análise; não comprova que aquele evento ocorreu nos dados reais.

## Efeito sobre divergências e competência

Uma taxa errada numa venda e seu estorno pode se anular na cobrança líquida. A tabela abaixo conserva as duas linhas e seus respectivos arquivos, sem transformar cancelamento econômico em exclusão do rastro. O documento registra chegada às 13h10 e confirma a inclusão na RN-65. Data de recebimento não altera a competência definida pela data_movimento (D-14).

In [4]:
pares=[]
for r in lote:
    if r['tipo_lancamento']!='ESTORNO' or len(indice.get(r['id_referencia'],[]))!=1: continue
    orig=indice[r['id_referencia']][0]
    taxa_o,_=contrato_para(orig); taxa_a,_=contrato_para(r)
    if taxa_o is not None and taxa_a is not None:
        do=centavos(orig['valor_bruto']*(orig['taxa_aplicada_pct']/100-taxa_o))
        da=centavos(r['valor_bruto']*(r['taxa_aplicada_pct']/100-taxa_a))
        pares.append(dict(venda=orig['id_transacao'],estorno=r['id_transacao'],
            divergencia_venda=do,divergencia_estorno=da,divergencia_liquida_par=do+da,
            cancela_bruto=orig['valor_bruto']==-r['valor_bruto']))
display(pd.DataFrame(pares))
print('Competências dos ajustes:',Counter(r['data_movimento'].strftime('%Y-%m') for r in lote))
print('Competências das operações referenciadas:',Counter(r['data_venda_original'].strftime('%Y-%m') for r in lote if r['data_venda_original']))
# Reversões contábeis não implicam desfazer TED/PIX: não há identificador bancário neste lote.
print('Sem referência bancária no layout:',not any(k in cols_lote for k in ['FITID','Documento']))


,venda,estorno,divergencia_venda,divergencia_estorno,divergencia_liquida_par,cancela_bruto
0,TX88431689,TX88446992,0.00,-0.00,0.00,True
1,TX88413573,TX88447011,0.00,-0.00,0.00,True
2,TX88438558,TX88447030,0.00,-0.00,0.00,True
3,TX88425142,TX88447044,0.00,-0.00,0.00,True
4,TX88430977,TX88447057,0.00,-0.00,0.00,True
5,TX88424537,TX88447070,0.00,-0.00,0.00,True
6,TX88422829,TX88447087,0.00,-0.00,0.00,True
7,TX88416714,TX88447093,0.00,-0.00,0.00,True
8,TX88428997,TX88447109,0.00,-0.00,0.00,True
9,TX88432148,TX88447140,0.00,-0.00,0.00,True


Competências dos ajustes: Counter({'2026-09': 14})
Competências das operações referenciadas: Counter({'2026-09': 13})
Sem referência bancária no layout: True


## Pendências e rastro

A RN-65 autoriza incorporar o adicional ao fechamento. Não existe mais pendência por falta de regra numerada ou pela chegada tardia em si. Pendências de negócio reais não tornam automaticamente o resultado provisório (D-13); a liberação permanece com o financeiro (RN-60/RN-61).

O rastro da comissão usa a contribuição de comissão de cada linha, não o bruto ou a taxa observada. Divergências usam sua própria contribuição. A referência à operação original é evidência, sem duplicar sua contribuição no ajuste. Mesmo ID em dois arquivos conta uma vez e exige pendência (RN-65); conteúdo conflitante deve ser bloqueado até resolução.

In [5]:
pendencias_gerais=[]  # RN-65 confirma a inclusão; não há bloqueio pela chegada tardia.
display(pd.DataFrame(pendencias)); display(pd.DataFrame(pendencias_gerais))
# Demonstração em memória, sem gravar CSV final.
def id_estavel(*partes): return hashlib.sha256('|'.join(partes).encode()).hexdigest()[:24]
rastro=[]; saidas=[]
for ec in sorted({r['ec'] for r in resultados}):
    grupo=[r for r in resultados if r['ec']==ec and r['comissao_incremental'] is not None]
    if not grupo: continue
    competencias={next(x for x in lote if x['id_transacao']==r['id_transacao'])['data_movimento'].strftime('%Y-%m') for r in grupo}
    for competencia in sorted(competencias):
        linhas=[r for r in grupo if next(x for x in lote if x['id_transacao']==r['id_transacao'])['data_movimento'].strftime('%Y-%m')==competencia]
        sid=id_estavel('SIMULACAO_INCREMENTAL_CHECKOUT',ec,competencia,hashes[AJUSTES])
        saidas.append({'id_saida':sid,'ec':ec,'valor':total(r['comissao_incremental'] for r in linhas)})
        for r in linhas: rastro.append(dict(id_saida=sid,arquivo_origem=r['arquivo_origem'],ref_origem=r['ref_origem'],valor=r['comissao_incremental']))
for s in saidas: assert total(r['valor'] for r in rastro if r['id_saida']==s['id_saida'])==s['valor']
display(pd.DataFrame(rastro))
print('Origens referenciadas (evidência, sem contribuição financeira duplicada):')
display(refs[['id_transacao','id_referencia','arquivo_referenciado']])


""


""


,id_saida,arquivo_origem,ref_origem,valor
0,59dec962b4dc165381b74701,06_orbe_ajustes_tardios_set2026.csv,TX88447087,-327.73
1,59dec962b4dc165381b74701,06_orbe_ajustes_tardios_set2026.csv,TX88447140,-61.28
2,59dec962b4dc165381b74701,06_orbe_ajustes_tardios_set2026.csv,AN88447152,-967.69
3,59dec962b4dc165381b74701,06_orbe_ajustes_tardios_set2026.csv,AN88447178,-875.49
4,1ad0d9c8e876a4b1df8979a9,06_orbe_ajustes_tardios_set2026.csv,TX88446992,-29.71
5,edfa9ba33afa2ff42310ec99,06_orbe_ajustes_tardios_set2026.csv,TX88447057,-124.36
6,edfa9ba33afa2ff42310ec99,06_orbe_ajustes_tardios_set2026.csv,TX88447109,-87.65
7,0753bb0284610032c7e17f7b,06_orbe_ajustes_tardios_set2026.csv,TX88447030,-14.00
8,8a062128a45153cceb251e08,06_orbe_ajustes_tardios_set2026.csv,TX88447225,174.81
9,7920d2437ccb097c0356b76a,06_orbe_ajustes_tardios_set2026.csv,TX88447011,-6.27


Origens referenciadas (evidência, sem contribuição financeira duplicada):


,id_transacao,id_referencia,arquivo_referenciado
0,TX88446992,TX88431689,01_orbe_relatorio_comissoes_set2026.csv
1,TX88447011,TX88413573,01_orbe_relatorio_comissoes_set2026.csv
2,TX88447030,TX88438558,01_orbe_relatorio_comissoes_set2026.csv
3,TX88447044,TX88425142,01_orbe_relatorio_comissoes_set2026.csv
4,TX88447057,TX88430977,01_orbe_relatorio_comissoes_set2026.csv
5,TX88447070,TX88424537,01_orbe_relatorio_comissoes_set2026.csv
6,TX88447087,TX88422829,01_orbe_relatorio_comissoes_set2026.csv
7,TX88447093,TX88416714,01_orbe_relatorio_comissoes_set2026.csv
8,TX88447109,TX88428997,01_orbe_relatorio_comissoes_set2026.csv
9,TX88447140,TX88432148,01_orbe_relatorio_comissoes_set2026.csv


## Verificações de reprocessamento

O ensaio abaixo classifica repetições sem eliminá-las das entradas brutas. Reentregar o mesmo lote não deve adicionar contribuição financeira; mesmo ID com conteúdo alterado precisa bloquear a consolidação até decisão. Arquivos idênticos com outro nome também precisam ser reconhecidos por hash no motor.

In [6]:
def consolidar(fontes):
    unicos={}; reenvios=[]; conflitos=[]
    for fonte in fontes:
        for r in fonte:
            anterior=unicos.get(r['id_transacao'])
            if anterior is None: unicos[r['id_transacao']]=r
            elif assinatura(anterior)==assinatura(r): reenvios.append(r)
            else: conflitos.append(r)
    return list(unicos.values()),reenvios,conflitos
primeiro,rep1,con1=consolidar([base,lote])
segundo,rep2,con2=consolidar([base,lote,lote])
assert {r['id_transacao'] for r in primeiro}=={r['id_transacao'] for r in segundo}
assert not con1 and not con2
assert total(r['valor_bruto'] for r in primeiro)==total(r['valor_bruto'] for r in segundo)
assert len(primeiro)==len(base)+len(lote)-len(rep1)
for k in campos_dinheiro:
    assert total(r[k] for r in primeiro)==total(r[k] for r in base+lote)-total(r[k] for r in rep1)
assert all(hashlib.sha256(p.read_bytes()).hexdigest()==hashes[p] for p in FONTES)
assert all(r['arquivo_origem']==AJUSTES.name and r['ref_origem']==r['id_transacao'] for r in lote)
assert len(resultados)==len(lote)
assert all(r['comissao_incremental'] is not None or any(p['ref_origem']==r['id_transacao'] for p in pendencias) for r in resultados)
print('Linhas originais:',len(base),'| linhas adicionais:',len(lote),'| consolidadas:',len(primeiro))
print('Reenvios reconhecidos na segunda entrega:',len(rep2),'| conflitos:',len(con2))
print('Conservação de bruto/taxa/líquido, rastro e hashes: OK')

referenciados=[x for x in referencias if x['id_referencia']]
assert all(x['qtd_origens']==1 and x['mesmo_ec'] and x['tipo_coerente'] and x['data_coerente'] for x in referenciados)
assert all(x['bruto_oposto'] and x['taxa_oposta'] and x['liquido_oposto'] for x in referenciados)
assert not any(x['excede_original'] for x in consumos)
print('Referências, estornos integrais e limites acumulados: OK')


Linhas originais: 1943 | linhas adicionais: 14 | consolidadas: 1957
Reenvios reconhecidos na segunda entrega: 14 | conflitos: 0
Conservação de bruto/taxa/líquido, rastro e hashes: OK
Referências, estornos integrais e limites acumulados: OK


## Detecção por layout e regras sem ocorrência nova

O ensaio reconhece arquivos Orbe pelo conjunto de colunas, sem depender de nome, e preserva diagnóstico de fontes com cabeçalho incompatível. Os cenários sintéticos abaixo validam RN-21/RN-22/RN-23 e a decisão de pendência em reenvio; seus valores são ilustrativos e não entram nos totais reais.

In [7]:
def reconhecer_orbe(conteudo):
    cabecalho=next(csv.reader(io.StringIO(conteudo.lstrip('\ufeff')),delimiter=';'),[])
    return len(cabecalho)==len(cols_base) and set(cabecalho)==set(cols_base)
fontes_orbe=[]
for arq in sorted(PASTA.iterdir()):
    if not arq.is_file() or arq.suffix.lower()!='.csv': continue
    raw=arq.read_bytes()
    try: txt=raw.decode('utf-8-sig')
    except UnicodeDecodeError: txt=raw.decode('cp1252')
    if reconhecer_orbe(txt): fontes_orbe.append(arq.name)
assert ORIGINAL.name in fontes_orbe and AJUSTES.name in fontes_orbe
assert reconhecer_orbe(AJUSTES.read_text(encoding='utf-8-sig'))
assert not reconhecer_orbe('campo_invalido;outro\n1;2')
display(pd.DataFrame({'fontes_orbe_por_cabecalho':fontes_orbe}))

def divergencia(bruto,aplicada,contratada):
    return aplicada!=contratada, centavos(bruto*(aplicada-contratada))
# Cenários sintéticos: taxa maior, seu estorno e taxa menor.
venda=divergencia(Decimal('100'),Decimal('0.03'),Decimal('0.02'))
estorno=divergencia(Decimal('-100'),Decimal('0.03'),Decimal('0.02'))
menor=divergencia(Decimal('100'),Decimal('0.01'),Decimal('0.02'))
assert venda[0] and estorno[0] and estorno[1]<0
assert venda[1]+estorno[1]==0
assert menor[0] and menor[1]<0
assert divergencia(Decimal('100'),Decimal('0.02'),Decimal('0.02'))==(False,Decimal('0.00'))
# RN-65: reenvio conta uma vez e gera detalhe de pendência.
pendencias_reenvio=[{'tipo':'ID_ORBE_REPETIDO','ref_origem':r['ref_origem'],
    'arquivo_origem':r['arquivo_origem'],'valor_brl':r['valor_bruto'],
    'motivo':'ID reenviado: conta uma vez (RN-65)','proximo_passo':'Revisar reenvio com a Orbe',
    'quem_decide':'FINANCEIRO'} for r in rep2]
assert len(pendencias_reenvio)==len(lote)
print('RN-21 / RN-22 / RN-23: cenários sintéticos OK')
print('RN-65: pendências no ensaio de reenvio:',len(pendencias_reenvio))
print('Divergências reais novas:',sum(r['divergencia_taxa_incremental'] is not None and r['divergencia_taxa_incremental']!=0 for r in resultados))


,fontes_orbe_por_cabecalho
0,01_orbe_relatorio_comissoes_set2026.csv
1,06_orbe_ajustes_tardios_set2026.csv


RN-21 / RN-22 / RN-23: cenários sintéticos OK
RN-65: pendências no ensaio de reenvio: 14
Divergências reais novas: 0


## Requisitos para o motor de checkout

1. Descobrir todos os relatórios Orbe compatíveis na pasta de entrada pelo layout e tipo de fonte, sem depender de um único nome ou sufixo mensal. Rejeitar schema incompatível com diagnóstico.
2. Registrar arquivo, hash, horário de ingestão e versão do fechamento. Resolver identidade por fonte + id_transacao; preservar todas as ocorrências no histórico de ingestão. Reenvio idêntico é idempotente; conteúdo conflitante exige revisão.
3. Consolidar os arquivos antes de resolver referências. Procurar a origem em todos os lotes e, quando necessário, em histórico de competência anterior. Validar EC, tipo, datas, modalidade, parcelas e limites acumulados.
4. Selecionar contrato por identidade aprovada e vigência. Estorno usa data da venda original; reversão precisa anular somente a antecipação referenciada. Não usar taxa aplicada como fallback.
5. Calcular com Decimal e arredondamento aprovado por linha. Recalcular comissões e divergências líquidas, sem misturar devolução comercial com cobrança ao parceiro. Reavaliar notas positivas por lojista após os ajustes, inclusive mudanças de sinal.
6. Reprocessar a competência afetada por inteiro ou aplicar deltas com controle de versão. Nunca somar o mesmo adicional a um fechamento que já o inclui. Não somar o delta a números de referência fixos.
7. Gerar pendências com transações detalhadas e responsabilidade. O financeiro decide reabertura, revisão de notas/contestações e liberação; o comercial decide contrato/cadastro.
8. Validar que toda entrada tem destino e que cada saída fecha com seu rastro antes de substituir arquivos finais de forma atômica. Separar rastros financeiros de vínculos documentais com operações anteriores.

Este notebook não recalcula o fechamento completo, não aprova emissão de notas e não modifica `src/`. A disponibilidade do novo arquivo muda a base de checkout; ela não cria, por si só, novos movimentos nos extratos bancários.

In [8]:
impacto=total(r['comissao_incremental'] for r in resultados if r['comissao_incremental'] is not None)
achados=[
    {'achado':'Lote adicional','evidencia':f"{len(lote)} linhas; tipos: {dict(Counter(r['tipo_lancamento'] for r in lote))}", 'impacto':'Amplia a base de checkout','recomendacao':'Ingestão de múltiplos arquivos'},
    {'achado':'Identidade','evidencia':f'{len(repetidos)} IDs repetidos/conflitantes na união','impacto':f'{len(primeiro)} movimentos consolidados','recomendacao':'Idempotência por identidade e conteúdo'},
    {'achado':'Referências','evidencia':f"{sum(x['qtd_origens']==1 for x in referencias)} origens únicas localizadas",'impacto':'Permite validar estornos e reversões','recomendacao':'Resolver origem antes de calcular'},
    {'achado':'Taxa observada da Orbe','evidencia':str(total(r['valor_taxa'] for r in lote)),'impacto':'Valor observado; não equivale automaticamente à comissão','recomendacao':'Manter separado do cálculo contratual'},
    {'achado':'Comissão incremental','evidencia':str(impacto),'impacto':f"Cálculo contratual de {sum(r['comissao_incremental'] is not None for r in resultados)} linhas, segundo documento final do repositório",'recomendacao':'Recalcular notas e saldos por lojista antes da liberação'},
    {'achado':'Pendências dos dados','evidencia':str(len(pendencias)),'impacto':'Linhas sem cálculo seguro permanecem fora da soma calculável','recomendacao':'Resolver com comercial/financeiro'},
    {'achado':'RN numeradas e aprovação','evidencia':'RN-13, RN-14, RN-20 a RN-23 e RN-65 disponíveis no repositório','impacto':'Inclusão do adicional confirmada; liberação continua com financeiro','recomendacao':'Registrar versão das regras e aprovador no manifesto'},
    {'achado':'Rastro incremental','evidencia':f'{len(rastro)} contribuições verificadas','impacto':'Soma por saída confere','recomendacao':'Preservar arquivo adicional + id_transacao e vínculos originais'}]
display(pd.DataFrame(achados))


,achado,evidencia,impacto,recomendacao
0,Lote adicional,"14 linhas; tipos: {'ESTORNO': 10, 'REVERSAO AN...",Amplia a base de checkout,Ingestão de múltiplos arquivos
1,Identidade,0 IDs repetidos/conflitantes na união,1957 movimentos consolidados,Idempotência por identidade e conteúdo
2,Referências,13 origens únicas localizadas,Permite validar estornos e reversões,Resolver origem antes de calcular
3,Taxa observada da Orbe,-3304.32,Valor observado; não equivale automaticamente ...,Manter separado do cálculo contratual
4,Comissão incremental,-3304.39,"Cálculo contratual de 14 linhas, segundo docum...",Recalcular notas e saldos por lojista antes da...
5,Pendências dos dados,0,Linhas sem cálculo seguro permanecem fora da s...,Resolver com comercial/financeiro
6,RN numeradas e aprovação,"RN-13, RN-14, RN-20 a RN-23 e RN-65 disponívei...",Inclusão do adicional confirmada; liberação co...,Registrar versão das regras e aprovador no man...
7,Rastro incremental,14 contribuições verificadas,Soma por saída confere,Preservar arquivo adicional + id_transacao e v...
